In [1]:
# Import libraries
import numpy as np                                      # Calculating tools
import pandas as pd                                     # Data manipulation
import scipy.stats as stats                             # Statistics tools
import matplotlib.pyplot as plt                         # Data visualization
import seaborn as sns                                   # Modern data visualization
import plotly.graph_objects as go                       # Interactive visualization

from sklearn.preprocessing import OrdinalEncoder, StandardScaler, LabelEncoder   # Encode categorical variable,  Data normalization
from sklearn.model_selection import train_test_split, StratifiedKFold                            # Create train, test set         
from sklearn import datasets                                                     # Some sample data sets 
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA      # LDA model
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA   # QDA model
from sklearn.neighbors import KNeighborsClassifier                               # kNN model 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
import statsmodels.api as sm                     # Statistical models
import statsmodels.formula.api as smf            # Statistical models with formula supports
from sklearn.feature_selection import VarianceThreshold
import ISLP 


In [2]:
train = pd.read_csv('Data/train.csv', index_col=0)
train_labels = pd.read_csv('Data/train_labels.csv', index_col=0)
test = pd.read_csv('Data/test.csv', index_col=0)

C:\Users\Source\AppData\Local\Temp\ipykernel_14468\2079969306.py:1: DtypeWarning: Columns (14742,14743,14749,14750,14751,14753,14754,14756,14759,14760,14761,14763,14768,14770,14774,14777,14780,14781,14782,14790,14792,14798,14799,14801,14805,14806,14807,14810,14812,14818,14819,14820,14825,14829,14833,14835,14837,14844,14845,14851,14852,14856,14862,14872,14877,14879,14880,14882,14883,14884,14887,14889,14891,14895,14897,14898,14901,14905,14907,14915,14917,14918,14919,14927,14929,14932,14933,14937,14938,14940,14943,14945,14948,14952,14955,14958,14959,14971,14973,14981,14997,14999) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('Data/train.csv', index_col=0)
C:\Users\Source\AppData\Local\Temp\ipykernel_14468\2079969306.py:3: DtypeWarning: Columns (14742,14743,14749,14750,14751,14753,14754,14756,14759,14760,14761,14763,14768,14770,14774,14777,14780,14781,14782,14790,14792,14798,14799,14801,14805,14806,14807,14810,14812,14818,14819,14820,14825,

In [3]:
train.head()

,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var14991,Var14992,Var14993,Var14994,Var14995,Var14996,Var14997,Var14998,Var14999,Var15000
ID,,,,,,,,,,,,,,,,,,,,,
502504,0,0,0,0,6,0,0.0,0,0,0,...,NaN,NaN,Y562h9T,NaN,Q8_a,Xjzr,NaN,NaN,NaN,NaN
197332,0,0,0,0,0,0,0.0,0,0,0,...,NaN,NaN,NyGn9rV,NaN,Q8_a,NBRvrWWx0Z,NaN,NaN,NaN,NaN
277830,0,0,0,0,0,0,0.0,0,0,0,...,NaN,NaN,Kbs60Tf,NaN,Q8_a,NBRvrWWx0Z,NaN,GD6M5hO,NaN,NaN
247676,0,0,0,0,0,0,0.0,0,0,0,...,NaN,NaN,QnVpiLG,NaN,KttQ,NBRvrWWx0Z,hv8kpZbKOENd20oRSYYeD3LZXLlVwZDpRutMj,rbDCepHN5tF_ZSV6,NaN,NaN
972007,0,0,0,0,0,0,0.0,0,0,0,...,NaN,NaN,RZyZF0b,NaN,KttQ,NBRvrWWx0Z,NaN,GD6M5hO,NaN,NaN


In [4]:
# Split the train (full) to train and validation
x_train, x_valid, y_train, y_valid = train_test_split(train, train_labels['Target_appetency'],
                                                      test_size=10000, stratify=train_labels['Target_appetency'], random_state=133)

In [5]:
x_train.shape
x_valid.shape
print(x_train.isna().sum().sum())
print(x_valid.isna().sum().sum())

10044110
5024336


# Data cleaning

## Only numerical values

In [6]:
train_num = x_train.iloc[:, :14740]
valid_num = x_valid.iloc[:, :14740]
test_num = test.iloc[:,:14740]

In [7]:
print(train_num.isna().sum().sum())
print(valid_num.isna().sum().sum())

5847601
2923307


Remove columns with 0 variance and where NAs qty <= 10% 

In [8]:
train_num = train_num.loc[:, train_num.var() != 0]
train_num = train_num.loc[:, train_num.isna().sum() <= len(train_num) * 0.1]


In [9]:
valid_num = valid_num[list(train_num.columns)]
test_num = test_num[list(train_num.columns)]

In [10]:
print(train_num.shape)
print(valid_num.shape)
print(test_num.shape)
print(train_num.isna().sum().sum())
print(valid_num.isna().sum().sum())
print(test_num.isna().sum().sum())

(20000, 12681)
(10000, 12681)
(20000, 12681)
1874
877
1885


In [11]:
#Get the columns with NAs
train_flags = train_num.isna().astype(np.uint8)
train_flags = train_flags.loc[:,train_flags.sum() >= 1]

valid_flags = valid_num[train_flags.columns].isna().astype(np.uint8)
valid_flags = valid_flags[list(train_flags.columns)]

test_flags = test_num[train_flags.columns].isna().astype(np.uint8)
test_flags = test_flags[list(train_flags.columns)]
#Change the name of the columns to flag_
train_flags.columns = [f'flag_{c}' for c in train_flags.columns]
valid_flags.columns = [f'flag_{c}' for c in valid_flags.columns]
test_flags.columns = [f'flag_{c}' for c in test_flags.columns]
#Calculate median of the columns with NAs
train_num = train_num.fillna(train_num.median())
valid_num = valid_num.fillna(valid_num.median())
test_num = test_num.fillna(test_num.median())

In [12]:
print(train_num.shape)
print(valid_num.shape)
print(test_num.shape)
print(train_num.isna().sum().sum())
print(valid_num.isna().sum().sum())
print(test_num.isna().sum().sum())

(20000, 12681)
(10000, 12681)
(20000, 12681)
0
0
0


In [13]:
#Standardize
scaler = StandardScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(train_num),
    columns=train_num.columns,
    index=train_num.index
)
valid_scaled = pd.DataFrame(
    scaler.transform(valid_num),
    columns=valid_num.columns,
    index=valid_num.index
)

test_scaled = pd.DataFrame(
    scaler.transform(test_num),
    columns=test_num.columns,
    index=test_num.index
)

In [14]:
train_num = pd.concat([train_scaled,train_flags],axis= 1)
valid_num = pd.concat([valid_scaled, valid_flags], axis=1)
test_num = pd.concat([test_scaled, test_flags], axis=1)

In [15]:
print(train_num.shape)
print(valid_num.shape)
print(test_num.shape)
print(train_num.isna().sum().sum())
print(valid_num.isna().sum().sum())
print(test_num.isna().sum().sum())

(20000, 12682)
(10000, 12682)
(20000, 12682)
0
0
0


## Categorical variables

In [20]:
train_cat = x_train.iloc[:, 14740:]
valid_cat = x_valid.iloc[:, 14740:]
test_cat = test.iloc[:,14740:]

In [21]:
print(train_cat.shape)
print(valid_cat.shape)
print(test_cat.shape)
print(train_cat.isna().sum().sum())
print(valid_cat.isna().sum().sum())
print(test_cat.isna().sum().sum())

(20000, 260)
(10000, 260)
(20000, 260)
4196509
2101029
4194581


In [22]:
#Remove columns where NAs qty <= 10% 
train_cat= train_cat.loc[:, train_cat.isna().sum() <= len(train_cat) * 0.1]
valid_cat = valid_cat[list(train_cat.columns)]
test_cat = test_cat[list(train_cat.columns)]

In [23]:
print(train_cat.shape)
print(valid_cat.shape)
print(test_cat.shape)
print(train_cat.isna().sum().sum())
print(valid_cat.isna().sum().sum())
print(test_cat.isna().sum().sum())

(20000, 26)
(10000, 26)
(20000, 26)
1659
800
1684


2 approaches: 

One hot encoding where number of categories < than 1000

Frequency encoding where number of categories > 1000. It creates a "hierarchy" based on frequency

In [24]:
#Hot encoding
cols = train_cat.columns[train_cat.nunique() < 1000]
# Train
train_cat[cols] = train_cat[cols].fillna('missing')
train_dummies = pd.get_dummies(train_cat[cols])
# Valid
valid_cat[cols] = valid_cat[cols].fillna('missing')
valid_dummies = pd.get_dummies(valid_cat[cols])
# Test
test_cat[cols] = test_cat[cols].fillna('missing') 
test_dummies = pd.get_dummies(test_cat[cols])
# same columns
valid_dummies = valid_dummies.reindex(columns=train_dummies.columns, fill_value=0)
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

train_cat = pd.concat([train_cat, train_dummies], axis=1)
valid_cat = pd.concat([valid_cat, valid_dummies], axis=1)
test_cat = pd.concat([test_cat, test_dummies], axis=1)
# Drop originales
train_cat.drop(columns=cols, inplace=True)
valid_cat.drop(columns=cols, inplace=True)
test_cat.drop(columns=cols, inplace=True)

In [25]:
print(train_cat.shape)
print(valid_cat.shape)
print(test_cat.shape)
print(train_cat.isna().sum().sum())
print(valid_cat.isna().sum().sum())
print(test_cat.isna().sum().sum())

(20000, 867)
(10000, 867)
(20000, 867)
298
122
288


In [26]:
train_cat = train_cat.fillna('missing')
valid_cat = valid_cat.fillna('missing')
test_cat = test_cat.fillna('missing')

In [27]:
#Frequency encoding
cols = train_cat.columns[train_cat.nunique() > 1000]

for col in cols:
    freq = train_cat[col].value_counts(normalize=True)
    
    train_cat[col + '_freq'] = train_cat[col].map(freq)
    valid_cat[col + '_freq'] = valid_cat[col].map(freq).fillna(0)
    test_cat[col + '_freq'] = test_cat[col].map(freq).fillna(0)

# Drop originales
train_cat.drop(columns=cols, inplace=True)
valid_cat.drop(columns=cols, inplace=True)
test_cat.drop(columns=cols, inplace=True)

In [28]:
print(train_cat.shape)
print(valid_cat.shape)
print(test_cat.shape)
print(train_cat.isna().sum().sum())
print(valid_cat.isna().sum().sum())
print(test_cat.isna().sum().sum())

(20000, 867)
(10000, 867)
(20000, 867)
0
0
0


In [29]:
#Remove columns with 0 variance
train_cat = train_cat.loc[:, train_cat.var() != 0]
valid_cat = valid_cat[list(train_cat.columns)]
test_cat = test_cat[list(train_cat.columns)]


In [30]:
print(train_cat.shape)
print(valid_cat.shape)
print(test_cat.shape)
print(train_cat.isna().sum().sum())
print(valid_cat.isna().sum().sum())
print(test_cat.isna().sum().sum())

(20000, 866)
(10000, 866)
(20000, 866)
0
0
0


In [31]:
print(train_num.shape)
print(valid_num.shape)
print(test_num.shape)
print(train_num.isna().sum().sum())
print(valid_num.isna().sum().sum())
print(test_num.isna().sum().sum())

(20000, 12682)
(10000, 12682)
(20000, 12682)
0
0
0


## Categorical + numerical 

Join text and numbers 

In [42]:
train2 = train_num.join(train_cat)
valid2 = valid_num.join(valid_cat)
test2 = test_num.join(test_cat)

In [45]:
print(train2.shape)
print(valid2.shape)
print(test2.shape)
print(train2.isna().sum().sum())

(20000, 13548)
(10000, 13548)
(20000, 13548)
0


### Correlation

In [44]:
#---------IT TOOK 180MIN TO RUN!!!!!!!
 
# Correlation matrix
#corr_matrix = train2.corr().abs()

# get only the upper site
#upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# columns with correlatio > 0.9
#threshold = 0.9
#high_corr_cols = [col for col in upper.columns if any(upper[col] > threshold)]

#print("Number of col:", len(high_corr_cols))
#print(high_corr_cols[:20])

In [ ]:
#threshold = 0.9
#drop = [col for col in upper.columns if any(upper[col] > threshold)]

#train2 = train2.drop(columns=drop)
#valid2 = valid2.drop(columns=drop)
#test2 = test2.drop(columns=drop)

#print(train2.shape)
#print(valid2.shape)
#print(test2.shape)

(20000, 11209)
(10000, 11209)
(20000, 11209)


In [ ]:
#pd.Series(drop).to_csv("high_corr_cols.csv", index=False)

###             

In [46]:
#Load drop columns from csv 
drop = pd.read_csv('high_corr_cols.csv')
drop = list(drop['0'])

In [47]:
train2 = train2.drop(columns=drop)
valid2 = valid2.drop(columns=drop)
test2 = test2.drop(columns=drop)

Feature selection

In [51]:
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
selector = SelectKBest(f_classif, k=18)
train_num2 = selector.fit_transform(train2,y_train)
valid_num2 = selector.transform(valid2)
test_num = selector.transform(test2)

c:\Users\Source\.conda\envs\py\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [5336] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
c:\Users\Source\.conda\envs\py\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


# Model

Logistic regression

In [68]:
model = LogisticRegression(penalty='l1',
                           solver= 'liblinear',
                            C=0.1)
n_col=18
model.fit(train_num2, y_train)
y_pred_train = pd.Series(model.predict_proba(train_num2)[:, 1], index=train_scaled.iloc[:,:n_col].index, name='pred_train')
y_pred_val = pd.Series(model.predict_proba(valid_num2)[:, 1], index=valid_scaled.iloc[:,:n_col].index, name='pred_val')
pred_test = pd.Series(model.predict_proba(test_num)[:, 1], index=test_scaled.iloc[:,:n_col].index, name='pred_test')

In [ ]:
auc_train = roc_auc_score(y_train, y_pred_train)
auc_val = roc_auc_score(y_valid, y_pred_val)

print(f'Train AUC: {auc_train:.4f}')
print(f'Validation AUC: {auc_val:.4f}')

#Fix overfitting 
#Select columns with categorical variables 
#Improve model by adjusting hyperparameters (We can use crossvalidation)
#Train AUC: 0.8484 with n_col = 50 and c = 0.8 best auc but more overfitting 
#Validation AUC: 0.8271

Train AUC: 0.8413
Validation AUC: 0.8280


In [70]:
# Submission
submission = pd.DataFrame({'Target_appetency': pred_test}, index=test.index)
submission.index.name = 'ID'
submission.to_csv('submission.csv')